In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
def convert_csv_to_series(data):
    pd_data=pd.read_csv(data)
    pd_series=pd_data.set_index("Date")
    cols = list(pd_series.columns)
    original_name=cols[0]
    cols[0] = 'Close'
    pd_series.columns = cols
    return pd_series,original_name

In [9]:
def convert_prices_to_returns(data):
    pd_series,name=convert_csv_to_series(data)[0],convert_csv_to_series(data)[1]
    pd_series["returns"]=pd_series['Close'] / pd_series['Close'].shift(1)
    return pd_series["returns"].dropna().to_frame(),name

In [10]:
sbi=convert_prices_to_returns("sbi_main.csv")[0]
nf_50=convert_prices_to_returns("nf_50_main.csv")[0]

In [11]:
tb_91=pd.read_excel("Treasury Bills Cut Off Yield Auction 91 Days.xlsx")

In [12]:
tb_91

,Month,yield
0,2025-01-01,6.5625
1,2025-02-01,6.4490
2,2025-03-01,6.5199
3,2025-04-01,5.9036
4,2025-05-01,5.6200
5,2025-06-01,5.4094
6,2025-07-01,5.3970
7,2025-08-01,5.5087
8,2025-09-01,5.4749
9,2025-10-01,5.4580


In [13]:
sbi.head()

,returns
Date,
2025-01-02,1.010086
2025-01-03,0.990265
2025-01-06,0.978573
2025-01-07,1.003027
2025-01-08,0.990241


In [14]:
nf_50.head()

,returns
Date,
2025-01-02,1.018774
2025-01-03,0.992397
2025-01-06,0.983807
2025-01-07,1.003889
2025-01-08,0.999201


In [18]:
def prepare_capm_data(df_stock, df_market, df_tb):
    """
    Combines daily stock/market gross returns with monthly T-Bill yields.
    """
    # 1. Ensure Index is Datetime
    df_stock.index = pd.to_datetime(df_stock.index)
    df_market.index = pd.to_datetime(df_market.index)
    df_tb['Month'] = pd.to_datetime(df_tb['Month'])
    df_tb = df_tb.set_index('Month')

   
    stock_ret = df_stock['returns'] - 1
    market_ret = df_market['returns'] - 1

    
    combined = pd.DataFrame({
        'Stock_Ret': stock_ret,
        'Market_Ret': market_ret
    }, index=df_stock.index)

    
    combined = combined.join(df_tb['yield'])

    
    combined['yield'] = combined['yield'].ffill()
    
    
    # Formula: (1 + Yield/100)^(1/252) - 1
    combined['Rf_daily'] = (1 + combined['yield']/100)**(1/252) - 1

    
    combined['Stock_Excess'] = combined['Stock_Ret'] - combined['Rf_daily']
    combined['Market_Excess'] = combined['Market_Ret'] - combined['Rf_daily']

    return combined.dropna()



In [19]:

final_df = prepare_capm_data(sbi, nf_50, tb_91)
final_df.head()

,Stock_Ret,Market_Ret,yield,Rf_daily,Stock_Excess,Market_Excess
Date,,,,,,
2025-02-01,-0.008927,-0.001117,6.449,0.000248,-0.009176,-0.001365
2025-02-03,-0.006593,-0.005157,6.449,0.000248,-0.006841,-0.005405
2025-02-04,0.023983,0.016189,6.449,0.000248,0.023735,0.015941
2025-02-05,-0.016876,-0.001809,6.449,0.000248,-0.017124,-0.002057
2025-02-06,-0.018015,-0.003923,6.449,0.000248,-0.018263,-0.004171


In [22]:
import statsmodels.api as sm

def run_capm_regression(df):
    """
    Performs OLS regression: (R_i - R_f) = alpha + beta * (R_m - R_f)
    Returns the fitted model and prints key metrics.
    """
    # 1. Define Dependent (Y) and Independent (X) variables
    y = df['Stock_Excess']
    X = df['Market_Excess']
    
    # 2. Add a constant to the independent variable for the intercept (Alpha)
    X = sm.add_constant(X)
    
    # 3. Fit the OLS model
    model = sm.OLS(y, X).fit()
    
    
    
    return model



In [23]:
capm_model = run_capm_regression(final_df)
capm_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           Stock_Excess   R-squared:                       0.318
Model:                            OLS   Adj. R-squared:                  0.315
Method:                 Least Squares   F-statistic:                     104.4
Date:                Sat, 11 Apr 2026   Prob (F-statistic):           2.28e-20
Time:                        00:23:03   Log-Likelihood:                 742.23
No. Observations:                 226   AIC:                            -1480.
Df Residuals:                     224   BIC:                            -1474.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0008      0.001      1.247      0.214      -0.000       0.002
Market_Excess     0.8492      0.083     10.219      0.000       0.685       1.013
==============================================================================
Omnibus:                       14.238   Durbin-Watson:                   2.041
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               23.551
Skew:                           0.361   Prob(JB):                     7.69e-06
Kurtosis:                       4.407   Cond. No.                         137.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""